# SDA with a pretrained ocean prior

This notebook samples the prior of a released checkpoint, then assimilates
SSH and SST observations from an OSSE tile with Moment Matching Posterior
Sampling (MMPS). Everything runs on one GPU in a few minutes.

Set `BUNDLE` to the directory holding `checkpoints/` and `osse_tiles/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

import obottle_starter as obs
from obottle_starter.inference.metrics import isotropic_psd, rmse
from obottle_starter.inference.osse import load_window, open_tile

BUNDLE = Path("..")  # contains checkpoints/ and osse_tiles/
CKPT = BUNDLE / "checkpoints" / "joint_ssh_sst_T5.pt"  # or ssh_only_T11.pt
TILE = BUNDLE / "osse_tiles" / "MidAtlantic_35N_50W.nc"
device = "cuda" if torch.cuda.is_available() else "cpu"

net, cfg = obs.load_checkpoint(CKPT, device)
names = obs.channel_names(cfg)
print(cfg.description)
print(f"channels {names}, {cfg.time_length} frames of {cfg.H}x{cfg.W}")

## The OSSE tile

Each tile is 336x336 cells at 4 km and 24 frames 12 hours apart. It holds the
llc4320 truth and the coverage of real instruments (SWOT + nadir altimeters
for SSH, ACSPO for SST). We cut a 128x128 window of `cfg.time_length` frames
out of the middle.

In [ ]:
tile = open_tile(TILE)
win = load_window(tile, names, cfg.time_length, t_start=0, stride=1, size=cfg.H)
print(win.times[0], "->", win.times[-1], "| centre lon", round(win.center_lon, 2))

fig, axes = plt.subplots(len(names), 2, figsize=(8, 4 * len(names)), squeeze=False)
for i, c in enumerate(names):
    axes[i, 0].imshow(
        win.truth[c][0], origin="lower", cmap="RdBu_r" if c == "SSH" else "inferno"
    )
    axes[i, 0].set_title(f"{c} truth, frame 0")
    axes[i, 1].imshow(win.masks[c].sum(0), origin="lower", cmap="Greys")
    axes[i, 1].set_title(f"{c} coverage, all frames ({100 * win.masks[c].mean():.0f}%)")
plt.tight_layout()

## Prior samples

With no observations `assimilate` samples the prior. The window's times and
longitude still condition the model (frame spacing, season, time of day).

In [ ]:
prior = obs.assimilate(
    net,
    cfg,
    times=win.times,
    center_lon=win.center_lon,
    truth=win.truth,
    masks={},
    n_members=4,
    n_batch=4,
    steps=64,
    seed=0,
)
fig, axes = plt.subplots(len(names), 4, figsize=(14, 3.5 * len(names)), squeeze=False)
for i, c in enumerate(names):
    for m in range(4):
        axes[i, m].imshow(
            prior.samples.sel(channel=c)[m, 0],
            origin="lower",
            cmap="RdBu_r" if c == "SSH" else "inferno",
        )
        axes[i, m].set_title(f"prior {c}, member {m}")
plt.tight_layout()

## Posterior samples

`sigma_y` is the observation-error standard deviation MMPS assumes, in
physical units (m for SSH, °C for SST). By default the synthetic
observations get Gaussian noise of that size. `mmps_iterations` sets the
number of Krylov iterations (one network backward pass each) per diffusion
step; more iterations fit the observations more closely at higher cost.

The "observations" panel below shows `post.observations`: the noisy
values the posterior is conditioned on, which the members should be
consistent with to within the observation error, not the noise-free truth.

In [ ]:
post = obs.assimilate(
    net,
    cfg,
    times=win.times,
    center_lon=win.center_lon,
    truth=win.truth,
    masks=win.masks,
    sigma_y={"SSH": 0.005, "SST": 0.1},
    n_members=8,
    n_batch=4,
    steps=64,
    mmps_iterations=4,
    seed=0,
)
mean, spread = post.samples.mean("sample"), post.samples.std("sample")
for c in names:
    err = rmse(mean.sel(channel=c), post.truth.sel(channel=c))
    print(
        f"{c}: RMSE per frame {np.round(err, 4)} | truth std "
        f"{float(post.truth.sel(channel=c).std()):.4f}"
    )

In [ ]:
t = cfg.time_length // 2
for c in names:
    cmap = "RdBu_r" if c == "SSH" else "inferno"
    truth = post.truth.sel(channel=c)[t]
    lims = dict(vmin=float(truth.min()), vmax=float(truth.max()))
    # The values the posterior is conditioned on: truth plus instrument noise
    # on the mask, NaN elsewhere (not the noise-free truth).
    observed = post.observations.sel(channel=c)[t]
    panels = [
        (truth, "truth", lims),
        (observed, "observations (noisy)", lims),
        (post.samples.sel(channel=c)[0, t], "member 0", lims),
        (mean.sel(channel=c)[t], "ensemble mean", lims),
        (spread.sel(channel=c)[t], "ensemble spread", {}),
    ]
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, (field, title, kw) in zip(axes, panels):
        im = ax.imshow(
            field,
            origin="lower",
            cmap=cmap if title != "ensemble spread" else "viridis",
            **kw,
        )
        ax.set_title(f"{c} {title}, frame {t}")
        plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()

## Isotropic spectra

Radially averaged wavenumber spectra of the truth, the posterior members and
the posterior mean. Members should follow the truth to small scales; the
mean loses small-scale power where the observations do not constrain them.

In [ ]:
fig, axes = plt.subplots(1, len(names), figsize=(6 * len(names), 4), squeeze=False)
for ax, c in zip(axes[0], names):
    for field, label in [
        (post.truth.sel(channel=c), "truth"),
        (post.samples.sel(channel=c), "members"),
        (mean.sel(channel=c), "ensemble mean"),
        (prior.samples.sel(channel=c), "prior"),
    ]:
        k, p = isotropic_psd(field.values, dx_km=4.0)
        ax.loglog(k, p, label=label)
    ax.set_xlabel("wavenumber (cycles/km)")
    ax.set_title(c)
    ax.legend()

## Things to try

* Observe only SSH (`masks={"SSH": win.masks["SSH"]}`) and see how much SST
  the joint model recovers.
* Use Bernoulli masks: `obottle_starter.inference.osse.random_masks(names, shape, 0.05)`.
* Change `sigma_y`, `mmps_iterations`, `steps`, or `mmps_solver`
  (`"gmres"`, `"cg"`, `"minres"`).
* Other tiles (`NorthAtlantic_48N_30W.nc`, `SubtropAtlantic_26N_44W.nc`),
  other windows (`origin=`, `t_start=`), or the SSH-only 11-frame model.
* Real observations: pass `y={channel: values}` instead of `truth=`.
* Any linear observation operator: see `obottle_starter.inference.sda.ObservationTerm`
  and `obottle_starter.inference.sda.run_inference`.